In [35]:
airports = pd.read_csv("../config/airports.csv")
import sys
import pandas as pd
sys.path.insert(0, "../src")

In [36]:

df = pd.read_csv("../data/raw/bts_2025_1.csv")
df

/var/folders/lq/8n2hx4hx6sq28_36c_vrzt2h0000gn/T/ipykernel_99797/2645219515.py:1: DtypeWarning: Columns (76: Div1TailNum, 77: Div2Airport, 84: Div2TailNum) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/bts_2025_1.csv")


,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,Unnamed: 109
0,2025,1,1,1,3,2025-01-01,AA,19805,AA,N104NN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025,1,1,2,4,2025-01-02,AA,19805,AA,N110AN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025,1,1,3,5,2025-01-03,AA,19805,AA,N106NN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025,1,1,4,6,2025-01-04,AA,19805,AA,N117AN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025,1,1,5,7,2025-01-05,AA,19805,AA,N104NN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
539742,2025,1,1,5,7,2025-01-05,NK,20416,NK,N682NK,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
539743,2025,1,1,6,1,2025-01-06,NK,20416,NK,N684NK,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
539744,2025,1,1,7,2,2025-01-07,NK,20416,NK,N662NK,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
539745,2025,1,1,8,3,2025-01-08,NK,20416,NK,N684NK,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
from pathlib import Path
cols = [
    "FlightDate", "Reporting_Airline", "Origin", "Dest",
    "CRSDepTime", "CRSArrTime", "CRSElapsedTime", "Distance",
    "ArrDelayMinutes", "Cancelled", "Diverted",
]

codes = ['ATL',
 'ORD',
 'DFW',
 'DEN',
 'LAX',
 'JFK',
 'SFO',
 'SEA',
 'LAS',
 'MCO',
 'CLT',
 'PHX',
 'MIA',
 'EWR',
 'IAH']
files = sorted(Path("../data/raw").glob("bts_*.csv"))
df = pd.concat((pd.read_csv(f, usecols=cols) for f in files), ignore_index=True)
df = df[df['Origin'].isin(codes) & df['Dest'].isin(codes)]
df = df[df["Cancelled"] == 0.0]
df = df[df["Diverted"] == 0.0]
df.dropna(subset=["ArrDelayMinutes"])

,FlightDate,Reporting_Airline,Origin,Dest,CRSDepTime,CRSArrTime,ArrDelayMinutes,Cancelled,Diverted,CRSElapsedTime,Distance
9,2024-11-21,OO,LAX,SFO,615,743,2.0,0.0,0.0,88.0,337.0
14,2024-11-21,OO,SFO,LAX,1814,1944,122.0,0.0,0.0,90.0,337.0
42,2024-11-21,OO,SEA,SFO,620,845,70.0,0.0,0.0,145.0,679.0
43,2024-11-21,OO,SFO,SEA,1000,1220,16.0,0.0,0.0,140.0,679.0
44,2024-11-21,OO,SEA,SFO,1745,2007,229.0,0.0,0.0,142.0,679.0
...,...,...,...,...,...,...,...,...,...,...,...
2811309,2025-03-03,NK,EWR,LAS,1648,1943,0.0,0.0,0.0,355.0,2227.0
2811398,2025-03-01,NK,DFW,LAS,726,833,0.0,0.0,0.0,187.0,1055.0
2811399,2025-03-02,NK,DFW,LAS,726,833,0.0,0.0,0.0,187.0,1055.0
2811400,2025-03-03,NK,DFW,LAS,726,833,0.0,0.0,0.0,187.0,1055.0


In [38]:
df["is_delayed"] = (df["ArrDelayMinutes"] >= 15).astype(int)
df["is_delayed"].value_counts()


is_delayed
0    308673
1     75676
Name: count, dtype: int64

In [39]:
dep_str = df["CRSDepTime"].astype(int).astype(str).str.zfill(4)
is_2400 = dep_str == "2400"
dep_str = dep_str.where(~is_2400, "0000")

df["local_dep"] = pd.to_datetime(df["FlightDate"] + " " + dep_str, format="%Y-%m-%d %H%M")
df["local_dep"] = df["local_dep"] + pd.to_timedelta(is_2400.astype(int), unit="D")

In [40]:
tz_map = dict(zip(airports["code"], airports["tz"]))
df["origin_tz"] = df["Origin"].map(tz_map)

df["scheduled_dep_utc"] = df.groupby("origin_tz")["local_dep"].transform(
    lambda s: s.dt.tz_localize(s.name, nonexistent="shift_forward", ambiguous="NaT").dt.tz_convert("UTC")
)

In [41]:
df["scheduled_arr_utc"] = df["scheduled_dep_utc"] + pd.to_timedelta(df["CRSElapsedTime"], unit="m")


In [42]:
df[["scheduled_dep_utc", "scheduled_arr_utc"]].isna().sum()

scheduled_dep_utc    2
scheduled_arr_utc    2
dtype: int64

In [43]:
df

,FlightDate,Reporting_Airline,Origin,Dest,CRSDepTime,CRSArrTime,ArrDelayMinutes,Cancelled,Diverted,CRSElapsedTime,Distance,is_delayed,local_dep,origin_tz,scheduled_dep_utc,scheduled_arr_utc
9,2024-11-21,OO,LAX,SFO,615,743,2.0,0.0,0.0,88.0,337.0,0,2024-11-21 06:15:00,America/Los_Angeles,2024-11-21 14:15:00+00:00,2024-11-21 15:43:00+00:00
14,2024-11-21,OO,SFO,LAX,1814,1944,122.0,0.0,0.0,90.0,337.0,1,2024-11-21 18:14:00,America/Los_Angeles,2024-11-22 02:14:00+00:00,2024-11-22 03:44:00+00:00
42,2024-11-21,OO,SEA,SFO,620,845,70.0,0.0,0.0,145.0,679.0,1,2024-11-21 06:20:00,America/Los_Angeles,2024-11-21 14:20:00+00:00,2024-11-21 16:45:00+00:00
43,2024-11-21,OO,SFO,SEA,1000,1220,16.0,0.0,0.0,140.0,679.0,1,2024-11-21 10:00:00,America/Los_Angeles,2024-11-21 18:00:00+00:00,2024-11-21 20:20:00+00:00
44,2024-11-21,OO,SEA,SFO,1745,2007,229.0,0.0,0.0,142.0,679.0,1,2024-11-21 17:45:00,America/Los_Angeles,2024-11-22 01:45:00+00:00,2024-11-22 04:07:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2811309,2025-03-03,NK,EWR,LAS,1648,1943,0.0,0.0,0.0,355.0,2227.0,0,2025-03-03 16:48:00,America/New_York,2025-03-03 21:48:00+00:00,2025-03-04 03:43:00+00:00
2811398,2025-03-01,NK,DFW,LAS,726,833,0.0,0.0,0.0,187.0,1055.0,0,2025-03-01 07:26:00,America/Chicago,2025-03-01 13:26:00+00:00,2025-03-01 16:33:00+00:00
2811399,2025-03-02,NK,DFW,LAS,726,833,0.0,0.0,0.0,187.0,1055.0,0,2025-03-02 07:26:00,America/Chicago,2025-03-02 13:26:00+00:00,2025-03-02 16:33:00+00:00
2811400,2025-03-03,NK,DFW,LAS,726,833,0.0,0.0,0.0,187.0,1055.0,0,2025-03-03 07:26:00,America/Chicago,2025-03-03 13:26:00+00:00,2025-03-03 16:33:00+00:00


In [44]:
from pathlib import Path

Path("../data/interim").mkdir(parents=True, exist_ok=True)
df.to_parquet("../data/interim/flights.parquet", index=False)

In [45]:

df1 = df[df["Origin"] == "ATL"]
df1 = df1[df1["Dest"] == "EWR"]
df1["is_delayed"].mean()

np.float64(0.3145708582834331)

In [46]:
df

,FlightDate,Reporting_Airline,Origin,Dest,CRSDepTime,CRSArrTime,ArrDelayMinutes,Cancelled,Diverted,CRSElapsedTime,Distance,is_delayed,local_dep,origin_tz,scheduled_dep_utc,scheduled_arr_utc
9,2024-11-21,OO,LAX,SFO,615,743,2.0,0.0,0.0,88.0,337.0,0,2024-11-21 06:15:00,America/Los_Angeles,2024-11-21 14:15:00+00:00,2024-11-21 15:43:00+00:00
14,2024-11-21,OO,SFO,LAX,1814,1944,122.0,0.0,0.0,90.0,337.0,1,2024-11-21 18:14:00,America/Los_Angeles,2024-11-22 02:14:00+00:00,2024-11-22 03:44:00+00:00
42,2024-11-21,OO,SEA,SFO,620,845,70.0,0.0,0.0,145.0,679.0,1,2024-11-21 06:20:00,America/Los_Angeles,2024-11-21 14:20:00+00:00,2024-11-21 16:45:00+00:00
43,2024-11-21,OO,SFO,SEA,1000,1220,16.0,0.0,0.0,140.0,679.0,1,2024-11-21 10:00:00,America/Los_Angeles,2024-11-21 18:00:00+00:00,2024-11-21 20:20:00+00:00
44,2024-11-21,OO,SEA,SFO,1745,2007,229.0,0.0,0.0,142.0,679.0,1,2024-11-21 17:45:00,America/Los_Angeles,2024-11-22 01:45:00+00:00,2024-11-22 04:07:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2811309,2025-03-03,NK,EWR,LAS,1648,1943,0.0,0.0,0.0,355.0,2227.0,0,2025-03-03 16:48:00,America/New_York,2025-03-03 21:48:00+00:00,2025-03-04 03:43:00+00:00
2811398,2025-03-01,NK,DFW,LAS,726,833,0.0,0.0,0.0,187.0,1055.0,0,2025-03-01 07:26:00,America/Chicago,2025-03-01 13:26:00+00:00,2025-03-01 16:33:00+00:00
2811399,2025-03-02,NK,DFW,LAS,726,833,0.0,0.0,0.0,187.0,1055.0,0,2025-03-02 07:26:00,America/Chicago,2025-03-02 13:26:00+00:00,2025-03-02 16:33:00+00:00
2811400,2025-03-03,NK,DFW,LAS,726,833,0.0,0.0,0.0,187.0,1055.0,0,2025-03-03 07:26:00,America/Chicago,2025-03-03 13:26:00+00:00,2025-03-03 16:33:00+00:00


In [47]:
df

,FlightDate,Reporting_Airline,Origin,Dest,CRSDepTime,CRSArrTime,ArrDelayMinutes,Cancelled,Diverted,CRSElapsedTime,Distance,is_delayed,local_dep,origin_tz,scheduled_dep_utc,scheduled_arr_utc
9,2024-11-21,OO,LAX,SFO,615,743,2.0,0.0,0.0,88.0,337.0,0,2024-11-21 06:15:00,America/Los_Angeles,2024-11-21 14:15:00+00:00,2024-11-21 15:43:00+00:00
14,2024-11-21,OO,SFO,LAX,1814,1944,122.0,0.0,0.0,90.0,337.0,1,2024-11-21 18:14:00,America/Los_Angeles,2024-11-22 02:14:00+00:00,2024-11-22 03:44:00+00:00
42,2024-11-21,OO,SEA,SFO,620,845,70.0,0.0,0.0,145.0,679.0,1,2024-11-21 06:20:00,America/Los_Angeles,2024-11-21 14:20:00+00:00,2024-11-21 16:45:00+00:00
43,2024-11-21,OO,SFO,SEA,1000,1220,16.0,0.0,0.0,140.0,679.0,1,2024-11-21 10:00:00,America/Los_Angeles,2024-11-21 18:00:00+00:00,2024-11-21 20:20:00+00:00
44,2024-11-21,OO,SEA,SFO,1745,2007,229.0,0.0,0.0,142.0,679.0,1,2024-11-21 17:45:00,America/Los_Angeles,2024-11-22 01:45:00+00:00,2024-11-22 04:07:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2811309,2025-03-03,NK,EWR,LAS,1648,1943,0.0,0.0,0.0,355.0,2227.0,0,2025-03-03 16:48:00,America/New_York,2025-03-03 21:48:00+00:00,2025-03-04 03:43:00+00:00
2811398,2025-03-01,NK,DFW,LAS,726,833,0.0,0.0,0.0,187.0,1055.0,0,2025-03-01 07:26:00,America/Chicago,2025-03-01 13:26:00+00:00,2025-03-01 16:33:00+00:00
2811399,2025-03-02,NK,DFW,LAS,726,833,0.0,0.0,0.0,187.0,1055.0,0,2025-03-02 07:26:00,America/Chicago,2025-03-02 13:26:00+00:00,2025-03-02 16:33:00+00:00
2811400,2025-03-03,NK,DFW,LAS,726,833,0.0,0.0,0.0,187.0,1055.0,0,2025-03-03 07:26:00,America/Chicago,2025-03-03 13:26:00+00:00,2025-03-03 16:33:00+00:00


In [48]:
df["is_weekend"] = (df["local_dep"].dt.dayofweek >= 5).astype(int)
print(df.groupby("is_weekend")["is_delayed"].mean())
print(df.groupby(df["local_dep"].dt.day_name())["is_delayed"].mean())

is_weekend
0    0.188312
1    0.217883
Name: is_delayed, dtype: float64
local_dep
Friday       0.197096
Monday       0.201058
Saturday     0.203201
Sunday       0.231021
Thursday     0.218317
Tuesday      0.146000
Wednesday    0.172441
Name: is_delayed, dtype: float64


/var/folders/lq/8n2hx4hx6sq28_36c_vrzt2h0000gn/T/ipykernel_99797/555822418.py:1: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["is_weekend"] = (df["local_dep"].dt.dayofweek >= 5).astype(int)


In [49]:
from pathlib import Path
import pandas as pd
from fdf.weather import load_all_weather

flights = pd.read_parquet("../data/interim/flights.parquet")
weather = load_all_weather(
    Path("../config/airports.csv"),
    "2024-11-01",
    "2025-04-02",
    Path("../data/raw/weather"),
)

In [50]:
flights["dep_hour_utc"] = flights["scheduled_dep_utc"].dt.floor("h")

In [51]:
flights["arr_hour_utc"] = flights["scheduled_arr_utc"].dt.floor("h")

  

  

In [54]:
weather_cols = [
    "temp_c",
    "wind_speed_kmh",
    "wind_gusts_kmh",
    "precipitation_mm",
    "snowfall_cm",
    "cloud_cover_pct",
]

flights["dep_hour_utc"] = flights["scheduled_dep_utc"].dt.floor("h")
flights["arr_hour_utc"] = flights["scheduled_arr_utc"].dt.floor("h")

origin_weather = weather.rename(columns={c: f"origin_{c}" for c in weather_cols})
df = flights.merge(
    origin_weather,
    how="left",
    left_on=["Origin", "dep_hour_utc"],
    right_on=["airport", "time_utc"],
    validate="many_to_one",
).drop(columns=["airport", "time_utc"])

dest_weather = weather.rename(columns={c: f"dest_{c}" for c in weather_cols})
df = df.merge(
    dest_weather,
    how="left",
    left_on=["Dest", "arr_hour_utc"],
    right_on=["airport", "time_utc"],
    validate="many_to_one",
).drop(columns=["dep_hour_utc", "arr_hour_utc", "airport", "time_utc"])

print("Rows before:", len(flights), "| Rows after:", len(df))

Rows before: 384349 | Rows after: 384349


In [61]:
df[[f"origin_{c}" for c in weather_cols] + [f"dest_{c}" for c in weather_cols]].isna().mean()
df[df["origin_snowfall_cm"] > 0].head(5)

,FlightDate,Reporting_Airline,Origin,Dest,CRSDepTime,CRSArrTime,ArrDelayMinutes,Cancelled,Diverted,CRSElapsedTime,...,origin_wind_gusts_kmh,origin_precipitation_mm,origin_snowfall_cm,origin_cloud_cover_pct,dest_temp_c,dest_wind_speed_kmh,dest_wind_gusts_kmh,dest_precipitation_mm,dest_snowfall_cm,dest_cloud_cover_pct
23,2024-11-21,OO,ORD,JFK,1350,1659,60.0,0.0,0.0,129.0,...,67.0,0.3,0.14,100.0,9.7,10.8,24.1,0.4,0.0,100.0
468,2024-11-27,OO,DEN,LAX,1255,1434,18.0,0.0,0.0,159.0,...,20.2,0.1,0.07,99.0,16.3,10.9,29.5,0.0,0.0,100.0
728,2024-11-21,YX,ORD,JFK,1045,1405,87.0,0.0,0.0,140.0,...,47.9,1.8,1.26,100.0,9.5,13.7,36.0,0.9,0.0,100.0
820,2024-11-27,UA,DEN,PHX,745,946,34.0,0.0,0.0,121.0,...,22.3,0.7,0.49,100.0,16.4,3.6,9.7,0.0,0.0,100.0
833,2024-11-27,UA,DEN,MIA,1005,1604,27.0,0.0,0.0,239.0,...,27.7,0.2,0.14,100.0,26.3,10.0,25.9,0.0,0.0,0.0


In [62]:
df.groupby(pd.cut(df["origin_precipitation_mm"], [-0.1, 0, 2, 100]))["is_delayed"].mean()

origin_precipitation_mm
(-0.1, 0.0]     0.182417
(0.0, 2.0]      0.303136
(2.0, 100.0]    0.358007
Name: is_delayed, dtype: float64

In [63]:
df.to_parquet("../data/interim/flights_with_weather.parquet", index=False)